In [1]:
# ============================================================
# CELL 1 — INSTALL (Optional)
# ============================================================

# Uncomment if dependencies are missing
# !pip install pandas numpy xgboost scikit-learn

In [2]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import os
import json
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

print("Libraries loaded successfully.")
print(f"Pandas: {pd.__version__}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Libraries loaded successfully.
Pandas: 3.0.3
Timestamp: 2026-05-26 17:33:42


In [3]:
# ============================================================
# CELL 3 — CONFIGURATION
# ============================================================

CONFIG = {
    # Paths
    "feature_dataset_path": "../parquet_exports/feature_engineered_dataset.parquet",
    "gold_sla_path": "../parquet_exports/gold_sla_prediction_features.parquet",
    "export_dir": "../parquet_exports",
    "eval_dir": "../evaluation",
    "gold_export_path": "../parquet_exports/gold_ml_dataset.parquet",
    "dictionary_export_path": "../parquet_exports/gold_feature_dictionary.json",
    "quality_report_path": "../evaluation/gold_quality_report.txt",

    # Join
    "join_left_key": "ticket_pk",
    "join_right_key": "source_ticket_pk",
    "join_type": "left",

    # SLA columns to merge from gold dataset
    "sla_merge_cols": [
        "was_sla_breached",
        "priority_score",
        "ticket_age_hours",
        "resolution_time_hours",
        "followup_count",
        "avg_followup_content_length",
        "issue_complexity_score",
        "customer_tenure_months",
        "previous_tickets",
    ],

    # Feature validation thresholds
    "max_null_pct": 99.0,
    "min_variance": 1e-8,
    "max_outlier_zscore": 10.0,

    # Heuristic risk scoring
    "escalation_weights": {
        "retrieval_quality_score": 0.4,
        "text_complexity_score": 0.3,
        "special_char_ratio": 0.2,
        "digit_ratio": 0.1,
    },
    "escalation_threshold": 0.6,

    # Random state for reproducibility
    "random_state": 42,
}

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

CONFIGURATION
  feature_dataset_path: ../parquet_exports/feature_engineered_dataset.parquet
  gold_sla_path: ../parquet_exports/gold_sla_prediction_features.parquet
  export_dir: ../parquet_exports
  eval_dir: ../evaluation
  gold_export_path: ../parquet_exports/gold_ml_dataset.parquet
  dictionary_export_path: ../parquet_exports/gold_feature_dictionary.json
  quality_report_path: ../evaluation/gold_quality_report.txt
  join_left_key: ticket_pk
  join_right_key: source_ticket_pk
  join_type: left
  sla_merge_cols: ['was_sla_breached', 'priority_score', 'ticket_age_hours', 'resolution_time_hours', 'followup_count', 'avg_followup_content_length', 'issue_complexity_score', 'customer_tenure_months', 'previous_tickets']
  max_null_pct: 99.0
  min_variance: 1e-08
  max_outlier_zscore: 10.0
  escalation_weights: {'retrieval_quality_score': 0.4, 'text_complexity_score': 0.3, 'special_char_ratio': 0.2, 'digit_ratio': 0.1}
  escalation_threshold: 0.6
  random_state: 42


In [4]:
# ============================================================
# CELL 4 — LOGGING HELPERS
# ============================================================

def log(level, component, message):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] [{level}] [{component}] {message}")


def log_step(step_name):
    print(f"\n{'=' * 60}")
    print(f"  {step_name}")
    print(f"{'=' * 60}")


print("Logging helpers ready.")

Logging helpers ready.


In [5]:
# ============================================================
# CELL 5 — LOAD FEATURE ENGINEERED DATASET
# ============================================================

log_step("LOADING FEATURE ENGINEERED DATASET")

df_feat = None
try:
    t0 = time.time()
    df_feat = pd.read_parquet(CONFIG["feature_dataset_path"])
    elapsed = time.time() - t0
    log("OK", "DATA", f"Loaded {len(df_feat):,} rows, {df_feat.shape[1]} cols in {elapsed:.2f}s")
except FileNotFoundError:
    log("ERROR", "DATA", f"Dataset not found at {CONFIG['feature_dataset_path']}")
    log("ERROR", "DATA", "Run notebook 04_feature_engineering.ipynb first")
    raise
except Exception as e:
    log("ERROR", "DATA", f"Failed to load: {e}")
    raise

# Validate expected columns
EXPECTED_FEATURE_COLS = [
    "document_id", "ticket_pk", "retrieval_text_clean",
    "source_system_encoded", "similarity_method_encoded",
    "priority_encoded", "urgency_encoded", "impact_encoded",
    "text_word_count", "text_char_count", "avg_word_length",
    "unique_word_ratio", "uppercase_ratio", "digit_ratio",
    "special_char_ratio", "repetition_ratio",
    "text_complexity_score", "retrieval_quality_score",
    "corpus_quality_score", "similarity_confidence",
]

missing = [c for c in EXPECTED_FEATURE_COLS if c not in df_feat.columns]
if missing:
    log("ERROR", "VALIDATE", f"Missing expected columns: {missing}")
    raise ValueError(f"Feature dataset missing columns: {missing}")
log("OK", "VALIDATE", f"All {len(EXPECTED_FEATURE_COLS)} expected columns present")

print(f"\nColumns: {list(df_feat.columns)}")
df_feat.head(2)


  LOADING FEATURE ENGINEERED DATASET


[17:33:43] [OK] [DATA] Loaded 230,088 rows, 21 cols in 0.28s
[17:33:43] [OK] [VALIDATE] All 20 expected columns present

Columns: ['document_id', 'ticket_pk', 'retrieval_text_clean', 'metadata_json', 'source_system_encoded', 'similarity_method_encoded', 'priority_encoded', 'urgency_encoded', 'impact_encoded', 'text_word_count', 'text_char_count', 'avg_word_length', 'unique_word_ratio', 'uppercase_ratio', 'digit_ratio', 'special_char_ratio', 'repetition_ratio', 'text_complexity_score', 'retrieval_quality_score', 'corpus_quality_score', 'similarity_confidence']


,document_id,ticket_pk,retrieval_text_clean,metadata_json,source_system_encoded,similarity_method_encoded,priority_encoded,urgency_encoded,impact_encoded,text_word_count,text_char_count,avg_word_length,unique_word_ratio,uppercase_ratio,digit_ratio,special_char_ratio,repetition_ratio,text_complexity_score,retrieval_quality_score,corpus_quality_score,similarity_confidence
0,doc_0,2013_2,prio_high urg_critical impact_medium ticket_unknown_status resolution_overdu...,"{""ticket_pk"": ""2013_2"", ""source_system"": ""GLPI"", ""priority_encoded"": 4.0, ""u...",0,1,4.0,5.0,3.0,6,97,15.333333,1.0,0.0,0.0,0.072165,0.0,6.545333,0.516667,0.3,0.25
1,doc_1,2013_3,prio_medium urg_medium impact_medium ticket_unknown_status resolution_overdu...,"{""ticket_pk"": ""2013_3"", ""source_system"": ""GLPI"", ""priority_encoded"": 3.0, ""u...",0,1,3.0,3.0,3.0,6,97,15.333333,1.0,0.0,0.0,0.072165,0.0,6.545333,0.516667,0.3,0.25


In [6]:
# ============================================================
# CELL 6 — LOAD & MERGE GOLD SLA FEATURES
# ============================================================

log_step("LOADING & MERGING GOLD SLA FEATURES")

df_sla = None
try:
    t0 = time.time()
    df_sla = pd.read_parquet(CONFIG["gold_sla_path"])
    elapsed = time.time() - t0
    log("OK", "GOLD", f"Loaded {len(df_sla):,} rows, {df_sla.shape[1]} cols in {elapsed:.2f}s")
except FileNotFoundError:
    log("ERROR", "GOLD", f"Gold SLA dataset not found at {CONFIG['gold_sla_path']}")
    raise
except Exception as e:
    log("ERROR", "GOLD", f"Failed to load: {e}")
    raise

# Validate merge columns
merge_cols = CONFIG["sla_merge_cols"].copy()
merge_cols.append(CONFIG["join_right_key"])
missing_gold = [c for c in merge_cols if c not in df_sla.columns]
if missing_gold:
    log("ERROR", "GOLD", f"Missing gold columns: {missing_gold}")
    raise ValueError(f"Gold SLA dataset missing columns: {missing_gold}")

# Merge on ticket_pk ↔ source_ticket_pk
t0 = time.time()
df_merged = df_feat.merge(
    df_sla[merge_cols],
    left_on=CONFIG["join_left_key"],
    right_on=CONFIG["join_right_key"],
    how=CONFIG["join_type"],
)
elapsed = time.time() - t0

# Drop redundant join key from right side
if CONFIG["join_right_key"] in df_merged.columns and CONFIG["join_right_key"] != CONFIG["join_left_key"]:
    df_merged.drop(columns=[CONFIG["join_right_key"]], inplace=True)

log("OK", "MERGE", f"Merged to {len(df_merged):,} rows in {elapsed:.2f}s")

# Validate merge quality
sla_nulls = df_merged["was_sla_breached"].isna().sum()
if sla_nulls > 0:
    log("WARN", "MERGE", f"{sla_nulls:,} rows without SLA match ({sla_nulls/len(df_merged)*100:.1f}%)")
else:
    log("OK", "MERGE", "All rows matched with SLA data")

print(f"\nMerged columns ({df_merged.shape[1]}): {list(df_merged.columns)}")
print(f"\nSLA breach distribution:")
print(df_merged["was_sla_breached"].value_counts().to_string())
print(f"  Null: {df_merged['was_sla_breached'].isna().sum()}")


  LOADING & MERGING GOLD SLA FEATURES
[17:33:43] [OK] [GOLD] Loaded 230,114 rows, 21 cols in 0.10s


[17:33:43] [OK] [MERGE] Merged to 230,088 rows in 0.25s
[17:33:43] [OK] [MERGE] All rows matched with SLA data

Merged columns (30): ['document_id', 'ticket_pk', 'retrieval_text_clean', 'metadata_json', 'source_system_encoded', 'similarity_method_encoded', 'priority_encoded', 'urgency_encoded', 'impact_encoded', 'text_word_count', 'text_char_count', 'avg_word_length', 'unique_word_ratio', 'uppercase_ratio', 'digit_ratio', 'special_char_ratio', 'repetition_ratio', 'text_complexity_score', 'retrieval_quality_score', 'corpus_quality_score', 'similarity_confidence', 'was_sla_breached', 'priority_score', 'ticket_age_hours', 'resolution_time_hours', 'followup_count', 'avg_followup_content_length', 'issue_complexity_score', 'customer_tenure_months', 'previous_tickets']

SLA breach distribution:
was_sla_breached
0    229177
1       911
  Null: 0


In [7]:
# ============================================================
# CELL 7 — DERIVE HEURISTIC FEATURES
# ============================================================
#
# All features in this cell are derived from text statistics ONLY.
# They do NOT use priority_encoded, urgency_encoded, impact_encoded,
# or was_sla_breached — zero target leakage by construction.
#
# Each feature is explicitly tagged as HEURISTIC/DERIVED.
# ============================================================

log_step("DERIVING HEURISTIC FEATURES")

df = df_merged.copy()
derived_features = []

# --- HEURISTIC: escalation_risk_score ---
# Composite of text-derived quality scores. Higher = more likely to need escalation.
# Formula: weighted combination of retrieval_quality, text_complexity, 
#          special_char_ratio, and digit_ratio (proxy for ticket formatting quality).
# No target columns accessed.
w = CONFIG["escalation_weights"]
df["escalation_risk_score"] = (
    df["retrieval_quality_score"].fillna(0) * w["retrieval_quality_score"]
    + (df["text_complexity_score"].fillna(0) / 10.0) * w["text_complexity_score"]
    + df["special_char_ratio"].fillna(0) * w["special_char_ratio"]
    + df["digit_ratio"].fillna(0) * w["digit_ratio"]
)
df["escalation_risk_score"] = df["escalation_risk_score"].clip(0, 1)
derived_features.append({
    "name": "escalation_risk_score",
    "type": "heuristic",
    "origin": "text-derived composite",
    "description": "Weighted composite of retrieval_quality_score, text_complexity_score/10, special_char_ratio, digit_ratio",
    "leakage_risk": "none",
})
log("OK", "HEURISTIC", f"escalation_risk_score: range [{df['escalation_risk_score'].min():.4f}, {df['escalation_risk_score'].max():.4f}]")

# --- HEURISTIC: is_predicted_high_risk ---
# Binary threshold on escalation_risk_score. Not derived from operational escalation data.
threshold = CONFIG["escalation_threshold"]
df["is_predicted_high_risk"] = (df["escalation_risk_score"] >= threshold).astype(int)
high_risk_pct = df["is_predicted_high_risk"].mean() * 100
derived_features.append({
    "name": "is_predicted_high_risk",
    "type": "heuristic",
    "origin": "threshold on escalation_risk_score >= " + str(threshold),
    "description": f"Binary flag: 1 if escalation_risk_score >= {threshold}, else 0",
    "leakage_risk": "none",
})
log("OK", "HEURISTIC", f"is_predicted_high_risk: {high_risk_pct:.1f}% high risk (threshold={threshold})")

# --- DERIVED: text_content_density ---
# Ratio of word count to character count — proxy for information density.
df["text_content_density"] = np.where(
    df["text_char_count"] > 0,
    df["text_word_count"] / df["text_char_count"],
    0.0,
)
derived_features.append({
    "name": "text_content_density",
    "type": "derived",
    "origin": "text_word_count / text_char_count",
    "description": "Information density proxy: words per character",
    "leakage_risk": "none",
})
log("OK", "DERIVED", f"text_content_density: range [{df['text_content_density'].min():.4f}, {df['text_content_density'].max():.4f}]")

# --- DERIVED: text_formatting_ratio ---
# Sum of formatting-related ratios. Higher = more non-standard text.
df["text_formatting_ratio"] = (
    df["uppercase_ratio"].fillna(0)
    + df["digit_ratio"].fillna(0)
    + df["special_char_ratio"].fillna(0)
)
derived_features.append({
    "name": "text_formatting_ratio",
    "type": "derived",
    "origin": "uppercase_ratio + digit_ratio + special_char_ratio",
    "description": "Aggregate non-standard character ratio",
    "leakage_risk": "none",
})
log("OK", "DERIVED", f"text_formatting_ratio: range [{df['text_formatting_ratio'].min():.4f}, {df['text_formatting_ratio'].max():.4f}]")

log("OK", "HEURISTIC", f"Total derived features added: {len(derived_features)}")
print(f"\nDerived feature summary:")
for f in derived_features:
    print(f"  {f['name']:30s} [{f['type']:9s}] {f['leakage_risk']}")


  DERIVING HEURISTIC FEATURES
[17:33:43] [OK] [HEURISTIC] escalation_risk_score: range [0.3772, 0.6757]
[17:33:43] [OK] [HEURISTIC] is_predicted_high_risk: 0.0% high risk (threshold=0.6)
[17:33:43] [OK] [DERIVED] text_content_density: range [0.0421, 0.2169]
[17:33:43] [OK] [DERIVED] text_formatting_ratio: range [0.0000, 0.0875]
[17:33:43] [OK] [HEURISTIC] Total derived features added: 4

Derived feature summary:
  escalation_risk_score          [heuristic] none
  is_predicted_high_risk         [heuristic] none
  text_content_density           [derived  ] none
  text_formatting_ratio          [derived  ] none


In [8]:
# ============================================================
# CELL 8 — FEATURE VALIDATION
# ============================================================

log_step("FEATURE VALIDATION")

validation_results = []

# Identify feature columns (excluding text/ID/metadata)
feature_cols = [c for c in df.columns if c not in [
    "document_id", "ticket_pk", "retrieval_text_clean",
    "metadata_json",
]]

print(f"Validating {len(feature_cols)} columns...\n")

# 1. Null percentage check
print("--- Null Analysis ---")
high_null_cols = []
for c in feature_cols:
    null_pct = df[c].isna().sum() / len(df) * 100
    status = "FAIL" if null_pct > CONFIG["max_null_pct"] else "PASS" if null_pct == 0 else "WARN"
    validation_results.append({"column": c, "check": "null_pct", "value": f"{null_pct:.1f}%", "status": status})
    if null_pct > 0:
        high_null_cols.append((c, null_pct))
        print(f"  {c:35s}: {null_pct:6.1f}% null  [{status}]")

if not high_null_cols:
    print(f"  All columns: 0% null  [PASS]")

# 2. Zero variance check
print("\n--- Variance Analysis ---")
zero_var_cols = []
for c in feature_cols:
    if np.issubdtype(df[c].dtype, np.number):
        var = df[c].var()
        if var < CONFIG["min_variance"] or pd.isna(var):
            zero_var_cols.append(c)
            validation_results.append({"column": c, "check": "variance", "value": f"{var:.2e}", "status": "FAIL"})
            print(f"  {c:35s}: var={var:.2e}  [FAIL]")

if not zero_var_cols:
    print(f"  All numeric columns: variance > threshold  [PASS]")

# 3. Duplicate column check
print("\n--- Duplicate Columns ---")
dup_cols = df.columns[df.columns.duplicated()].tolist()
if dup_cols:
    validation_results.append({"column": ", ".join(dup_cols), "check": "duplicates", "value": str(len(dup_cols)), "status": "FAIL"})
    print(f"  Duplicate columns found: {dup_cols}  [FAIL]")
else:
    print(f"  No duplicate columns  [PASS]")

# 4. Target label distributions
print("\n--- Target Distributions ---")
target_cols_real = ["priority_encoded", "urgency_encoded", "impact_encoded", "was_sla_breached"]
targets_present = [c for c in target_cols_real if c in df.columns]
for t in targets_present:
    nonnull = df[t].notna().sum()
    vc = df[t].value_counts().sort_index()
    print(f"  {t:25s}: {nonnull:6d} non-null / {len(df):,} total")
    if nonnull > 0 and nonnull < 100:
        print(f"    Values: {vc.to_dict()}")
    elif nonnull > 0:
        print(f"    Classes: {len(vc)}, Distribution: {vc.head(5).to_dict()}")

# Summary
n_fail = sum(1 for r in validation_results if r["status"] == "FAIL")
n_warn = sum(1 for r in validation_results if r["status"] == "WARN")
n_pass = sum(1 for r in validation_results if r["status"] == "PASS")

print(f"\nValidation summary: {n_pass} PASS, {n_warn} WARN, {n_fail} FAIL")

if n_fail > 0:
    log("WARN", "VALIDATE", f"Found {n_fail} validation failures — see details above")
else:
    log("OK", "VALIDATE", "All validation checks passed")

CONFIG["validation_results"] = validation_results


  FEATURE VALIDATION
Validating 30 columns...

--- Null Analysis ---
  priority_encoded                   :   99.3% null  [FAIL]
  urgency_encoded                    :   99.3% null  [FAIL]
  impact_encoded                     :   99.3% null  [FAIL]
  resolution_time_hours              :   12.5% null  [WARN]
  issue_complexity_score             :   13.1% null  [WARN]
  customer_tenure_months             :   13.1% null  [WARN]
  previous_tickets                   :   13.1% null  [WARN]

--- Variance Analysis ---
  uppercase_ratio                    : var=0.00e+00  [FAIL]



--- Duplicate Columns ---
  No duplicate columns  [PASS]

--- Target Distributions ---
  priority_encoded         :   1527 non-null / 230,088 total
    Classes: 4, Distribution: {2.0: 10, 3.0: 1318, 4.0: 158, 5.0: 41}
  urgency_encoded          :   1527 non-null / 230,088 total
    Classes: 4, Distribution: {2.0: 10, 3.0: 1325, 4.0: 132, 5.0: 60}
  impact_encoded           :   1527 non-null / 230,088 total
    Classes: 3, Distribution: {3.0: 1390, 4.0: 101, 5.0: 36}
  was_sla_breached         : 230088 non-null / 230,088 total
    Classes: 2, Distribution: {0: 229177, 1: 911}

Validation summary: 23 PASS, 4 WARN, 4 FAIL
[17:33:43] [WARN] [VALIDATE] Found 4 validation failures — see details above


In [9]:
# ============================================================
# CELL 9 — FEATURE DICTIONARY
# ============================================================

log_step("BUILDING FEATURE DICTIONARY")

# Define feature provenance for all columns
FEATURE_DICTIONARY = {
    # IDs & Keys
    "document_id": {"type": "string", "origin": "synthetic", "purpose": "Unique doc identifier for retrieval", "leakage_risk": "none"},
    "ticket_pk": {"type": "string", "origin": "real", "purpose": "Primary key from gold_ticket_similarity", "leakage_risk": "none"},
    "retrieval_text_clean": {"type": "string", "origin": "real", "purpose": "Cleaned text corpus for retrieval/embedding", "leakage_risk": "none"},

    # Encoded Categorical
    "source_system_encoded": {"type": "int", "origin": "derived", "purpose": "Label-encoded source system", "leakage_risk": "none"},
    "similarity_method_encoded": {"type": "int", "origin": "derived", "purpose": "Label-encoded similarity method", "leakage_risk": "none"},

    # Real Targets (from source)
    "priority_encoded": {"type": "float", "origin": "real", "purpose": "Priority level 2-5 from source", "leakage_risk": "target"},
    "urgency_encoded": {"type": "float", "origin": "real", "purpose": "Urgency level 2-5 from source", "leakage_risk": "target"},
    "impact_encoded": {"type": "float", "origin": "real", "purpose": "Impact level 3-5 from source", "leakage_risk": "target"},
    "was_sla_breached": {"type": "int", "origin": "real", "purpose": "SLA breach label from gold SLA dataset", "leakage_risk": "target"},

    # Gold SLA Features (merged)
    "priority_score": {"type": "int", "origin": "real", "purpose": "Priority score from gold SLA features", "leakage_risk": "none (parallel to priority_encoded)"},
    "ticket_age_hours": {"type": "float", "origin": "derived", "purpose": "Ticket age in hours from gold SLA", "leakage_risk": "none"},
    "resolution_time_hours": {"type": "float", "origin": "real", "purpose": "Resolution time from gold SLA", "leakage_risk": "none"},
    "followup_count": {"type": "int", "origin": "real", "purpose": "Number of followups from gold SLA", "leakage_risk": "none"},
    "avg_followup_content_length": {"type": "float", "origin": "real", "purpose": "Average followup length from gold SLA", "leakage_risk": "none"},
    "issue_complexity_score": {"type": "float", "origin": "heuristic", "purpose": "Issue complexity heuristic from gold SLA", "leakage_risk": "low"},
    "customer_tenure_months": {"type": "float", "origin": "real", "purpose": "Customer tenure from gold SLA", "leakage_risk": "none"},
    "previous_tickets": {"type": "float", "origin": "real", "purpose": "Previous ticket count from gold SLA", "leakage_risk": "none"},

    # Text Statistics (derived in Notebook 04)
    "text_word_count": {"type": "int", "origin": "derived", "purpose": "Word count of retrieval text", "leakage_risk": "none"},
    "text_char_count": {"type": "int", "origin": "derived", "purpose": "Character count of retrieval text", "leakage_risk": "none"},
    "avg_word_length": {"type": "float", "origin": "derived", "purpose": "Mean word length", "leakage_risk": "none"},
    "unique_word_ratio": {"type": "float", "origin": "derived", "purpose": "Unique/total word ratio", "leakage_risk": "none"},
    "uppercase_ratio": {"type": "float", "origin": "derived", "purpose": "Uppercase character ratio", "leakage_risk": "none"},
    "digit_ratio": {"type": "float", "origin": "derived", "purpose": "Digit character ratio", "leakage_risk": "none"},
    "special_char_ratio": {"type": "float", "origin": "derived", "purpose": "Special character ratio", "leakage_risk": "none"},
    "repetition_ratio": {"type": "float", "origin": "derived", "purpose": "Text repetition ratio", "leakage_risk": "none"},

    # Heuristic Composites (from Notebook 04)
    "text_complexity_score": {"type": "float", "origin": "heuristic", "purpose": "0.4*avg_wl + 0.4*uniq_ratio + 0.2*wc/100", "leakage_risk": "none"},
    "retrieval_quality_score": {"type": "float", "origin": "heuristic", "purpose": "Mean of corpus_quality, similarity_confidence, unique_word_ratio", "leakage_risk": "none"},

    # Real Source Scores (from gold_ticket_similarity)
    "corpus_quality_score": {"type": "float", "origin": "real", "purpose": "Corpus quality score from gold source", "leakage_risk": "none"},
    "similarity_confidence": {"type": "float", "origin": "real", "purpose": "Similarity confidence from gold source", "leakage_risk": "none"},

    # Heuristic Features (from Notebook 11)
    "escalation_risk_score": {"type": "float", "origin": "heuristic", "purpose": "Weighted composite of text-derived quality scores", "leakage_risk": "none"},
    "is_predicted_high_risk": {"type": "int", "origin": "heuristic", "purpose": f"Binary threshold on escalation_risk_score >= {CONFIG['escalation_threshold']}", "leakage_risk": "none"},

    # Derived Features (from Notebook 11)
    "text_content_density": {"type": "float", "origin": "derived", "purpose": "Information density: word_count / char_count", "leakage_risk": "none"},
    "text_formatting_ratio": {"type": "float", "origin": "derived", "purpose": "Aggregate non-standard char ratio", "leakage_risk": "none"},
}

print(f"Feature dictionary built: {len(FEATURE_DICTIONARY)} features documented")
print(f"\nFeature count by origin:")
origin_counts = {}
for k, v in FEATURE_DICTIONARY.items():
    o = v["origin"]
    origin_counts[o] = origin_counts.get(o, 0) + 1
for o, c in sorted(origin_counts.items()):
    print(f"  {o:10s}: {c}")

print(f"\nFeature count by leakage risk:")
leakage_counts = {}
for k, v in FEATURE_DICTIONARY.items():
    l = v["leakage_risk"]
    leakage_counts[l] = leakage_counts.get(l, 0) + 1
for l, c in sorted(leakage_counts.items()):
    print(f"  {l:20s}: {c}")

CONFIG["feature_dictionary"] = FEATURE_DICTIONARY


  BUILDING FEATURE DICTIONARY
Feature dictionary built: 33 features documented

Feature count by origin:
  derived   : 13
  heuristic : 5
  real      : 14
  synthetic : 1

Feature count by leakage risk:
  low                 : 1
  none                : 27
  none (parallel to priority_encoded): 1
  target              : 4


In [10]:
# ============================================================
# CELL 10 — EXPORT GOLD ML DATASET
# ============================================================

log_step("EXPORTING GOLD ML DATASET")

try:
    os.makedirs(CONFIG["export_dir"], exist_ok=True)
    os.makedirs(CONFIG["eval_dir"], exist_ok=True)

    t0 = time.time()

    # Export consolidated gold ML dataset
    df.to_parquet(CONFIG["gold_export_path"], index=False)
    elapsed = time.time() - t0
    log("OK", "EXPORT", f"Gold ML dataset saved to {CONFIG['gold_export_path']} ({elapsed:.2f}s)")

    # Export feature dictionary
    with open(CONFIG["dictionary_export_path"], "w") as f:
        json.dump(FEATURE_DICTIONARY, f, indent=2)
    log("OK", "EXPORT", f"Feature dictionary saved to {CONFIG['dictionary_export_path']}")

    # Export quality report
    with open(CONFIG["quality_report_path"], "w") as f:
        f.write("GOLD ML DATASET QUALITY REPORT\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Source features: {CONFIG['feature_dataset_path']}\n")
        f.write(f"Gold SLA features: {CONFIG['gold_sla_path']}\n")
        f.write(f"\n{'=' * 60}\n")
        f.write("DATASET OVERVIEW\n")
        f.write(f"{'=' * 60}\n")
        f.write(f"Total rows: {len(df):,}\n")
        f.write(f"Total columns: {df.shape[1]}\n")
        f.write(f"Feature columns: {len(feature_cols)}\n")
        f.write(f"Derived features: {len(derived_features)}\n")
        f.write(f"\n{'=' * 60}\n")
        f.write("TARGETS\n")
        f.write(f"{'=' * 60}\n")
        for t in targets_present:
            nonnull = df[t].notna().sum()
            f.write(f"\n{t}:\n")
            f.write(f"  Non-null: {nonnull} / {len(df)} ({nonnull/len(df)*100:.1f}%)\n")
            if nonnull > 0:
                vc = df[t].value_counts().sort_index()
                for val, cnt in vc.items():
                    f.write(f"  {val}: {cnt} ({cnt/len(df)*100:.1f}%)\n")

        f.write(f"\n{'=' * 60}\n")
        f.write("DERIVED FEATURES\n")
        f.write(f"{'=' * 60}\n")
        for feat in derived_features:
            f.write(f"\n{feat['name']}:\n")
            f.write(f"  Type: {feat['type']}\n")
            f.write(f"  Origin: {feat['origin']}\n")
            f.write(f"  Leakage: {feat['leakage_risk']}\n")
            f.write(f"  Description: {feat['description']}\n")

        f.write(f"\n{'=' * 60}\n")
        f.write("VALIDATION RESULTS\n")
        f.write(f"{'=' * 60}\n")
        for r in validation_results:
            f.write(f"  {r['column']:35s} | {r['check']:15s} | {r['value']:10s} | {r['status']}\n")

    log("OK", "EXPORT", f"Quality report saved to {CONFIG['quality_report_path']}")

    print(f"\nExported files:")
    print(f"  Gold ML dataset: {CONFIG['gold_export_path']}")
    print(f"  Feature dict:    {CONFIG['dictionary_export_path']}")
    print(f"  Quality report:  {CONFIG['quality_report_path']}")

except Exception as e:
    log("ERROR", "EXPORT", f"Export failed: {e}")
    raise


  EXPORTING GOLD ML DATASET


[17:33:44] [OK] [EXPORT] Gold ML dataset saved to ../parquet_exports/gold_ml_dataset.parquet (0.65s)
[17:33:44] [OK] [EXPORT] Feature dictionary saved to ../parquet_exports/gold_feature_dictionary.json
[17:33:44] [OK] [EXPORT] Quality report saved to ../evaluation/gold_quality_report.txt

Exported files:
  Gold ML dataset: ../parquet_exports/gold_ml_dataset.parquet
  Feature dict:    ../parquet_exports/gold_feature_dictionary.json
  Quality report:  ../evaluation/gold_quality_report.txt


In [11]:
# ============================================================
# CELL 11 — FINAL QUALITY REPORT
# ============================================================

log_step("GOLD ML DATASET QUALITY REPORT")

print(f"Dataset:")
print(f"  Rows:      {len(df):,}")
print(f"  Columns:   {df.shape[1]}")
print(f"  Features:  {len(feature_cols)}")
print(f"  Size:      {df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

print(f"\nTargets:")
for t in targets_present:
    nonnull = df[t].notna().sum()
    pct = nonnull / len(df) * 100
    print(f"  {t:30s}: {nonnull:7d} / {len(df):,} ({pct:.1f}% populated)")

print(f"\nFeature origins:")
for o, c in sorted(origin_counts.items()):
    print(f"  {o:12s}: {c} features")

print(f"\nLeakage risk:")
for l, c in sorted(leakage_counts.items()):
    print(f"  {l:20s}: {c} features")

print(f"\nDerived features added by this notebook:")
for feat in derived_features:
    col = df[feat["name"]]
    print(f"  {feat['name']:30s} [{feat['type']:9s}] range=[{col.min():.4f}, {col.max():.4f}], mean={col.mean():.4f}")

print(f"\nValidation:")
print(f"  {n_pass} PASS, {n_warn} WARN, {n_fail} FAIL")

print(f"\n{'=' * 60}")
print(f"  GOLD ML DATASET BUILD COMPLETE")
print(f"{'=' * 60}")


  GOLD ML DATASET QUALITY REPORT
Dataset:
  Rows:      230,088
  Columns:   34
  Features:  30
  Size:      164.4 MB

Targets:
  priority_encoded              :    1527 / 230,088 (0.7% populated)
  urgency_encoded               :    1527 / 230,088 (0.7% populated)
  impact_encoded                :    1527 / 230,088 (0.7% populated)
  was_sla_breached              :  230088 / 230,088 (100.0% populated)

Feature origins:
  derived     : 13 features
  heuristic   : 5 features
  real        : 14 features
  synthetic   : 1 features

Leakage risk:
  low                 : 1 features
  none                : 27 features
  none (parallel to priority_encoded): 1 features
  target              : 4 features

Derived features added by this notebook:
  escalation_risk_score          [heuristic] range=[0.3772, 0.6757], mean=0.3987
  is_predicted_high_risk         [heuristic] range=[0.0000, 1.0000], mean=0.0000
  text_content_density           [derived  ] range=[0.0421, 0.2169], mean=0.1628
  text_for

In [12]:
# ============================================================
# CELL 12 — LIMITATIONS & TRANSPARENCY REPORT
# ============================================================

log_step("LIMITATIONS & TRANSPARENCY")

print("=" * 60)
print("SYNTHETIC LABEL ANALYSIS")
print("=" * 60)
print("""
This notebook does NOT generate synthetic ML labels.

All targets in the Gold ML dataset are REAL labels from source data:
  - priority_encoded, urgency_encoded, impact_encoded: from gold_ticket_similarity
  - was_sla_breached: from gold_sla_prediction_features (fully populated)

New features added in this notebook are:
  - HEURISTIC: escalation_risk_score, is_predicted_high_risk
  - DERIVED: text_content_density, text_formatting_ratio

All new features are derived from text statistics only — zero target leakage.
""")

print("=" * 60)
print("KNOWN GOLD DATASET ISSUES")
print("=" * 60)
print("""
1. is_escalated (gold_sla_prediction_features): ALL ZEROS — broken label, excluded
2. anomaly_score, is_anomaly (gold_asset_failure_risk): ALL NULL — never computed
3. is_anomaly_if, is_anomaly_lof (gold_user_activity_anomalies): ALL NULL — never computed
4. private_followup_ratio (gold_sla_prediction_features): ALL ZEROS — likely feature bug
5. urgency_score, impact_score (gold_sla): 99.3% null — same sparsity as feature dataset
6. was_sla_breached: 0.4% positive rate — extreme class imbalance
""")

print("=" * 60)
print("TARGET LEAKAGE ANALYSIS")
print("=" * 60)
print("""
Verified: All derived/heuristic features are computed from text statistics only.
No target columns (priority_encoded, urgency_encoded, impact_encoded, was_sla_breached)
are used in any feature derivation.

Features verified leakage-free:
  - escalation_risk_score: uses retrieval_quality_score, text_complexity_score/10,
    special_char_ratio, digit_ratio — all text-derived, no targets
  - is_predicted_high_risk: threshold on escalation_risk_score — same derivation
  - text_content_density: word_count / char_count — pure text ratio
  - text_formatting_ratio: sum of uppercase/digit/special ratios — text formatting
""")

print("=" * 60)
print("NEXT STEPS FOR DOWNSTREAM CONSUMERS")
print("=" * 60)
print("""
1. Notebook 10 (triage): Can use gold_ml_dataset.parquet for training instead of
   feature_engineered_dataset.parquet — adds was_sla_breached as auxiliary label
2. Feature dictionary: gold_feature_dictionary.json provides schema + provenance
   for any downstream ML pipeline
3. SLA breach prediction: was_sla_breached (0.4% positive) is the only fully-populated
   binary label — suitable for imbalanced classification experiments
4. Escalation risk: escalation_risk_score provides a continuous heuristic baseline
   that future models should beat
""")


  LIMITATIONS & TRANSPARENCY
SYNTHETIC LABEL ANALYSIS

This notebook does NOT generate synthetic ML labels.

All targets in the Gold ML dataset are REAL labels from source data:
  - priority_encoded, urgency_encoded, impact_encoded: from gold_ticket_similarity
  - was_sla_breached: from gold_sla_prediction_features (fully populated)

New features added in this notebook are:
  - HEURISTIC: escalation_risk_score, is_predicted_high_risk
  - DERIVED: text_content_density, text_formatting_ratio

All new features are derived from text statistics only — zero target leakage.

KNOWN GOLD DATASET ISSUES

1. is_escalated (gold_sla_prediction_features): ALL ZEROS — broken label, excluded
2. anomaly_score, is_anomaly (gold_asset_failure_risk): ALL NULL — never computed
3. is_anomaly_if, is_anomaly_lof (gold_user_activity_anomalies): ALL NULL — never computed
4. private_followup_ratio (gold_sla_prediction_features): ALL ZEROS — likely feature bug
5. urgency_score, impact_score (gold_sla): 99.3% nul